# 3장 2강 · 모델 오류 분석과 성능 개선

**실습문제**

작은 동물을 놓치는 모델의 입력을 키우면 같은 사진에서 무엇이 달라질까요? 가중치를 먼저 실제로 학습한 뒤, 두 입력 크기를 같은 원본에 적용합니다. 전체 정답 수가 바뀐 이유를 개별 객체 번호까지 거슬러 올라가 확인합니다.

| 구분 | 수행할 일 |
|---|---|
| 필수 1 | 학습 기록 확인 → 예측과 정답 연결 → 오류 유형별 집계 → 작은 객체에 관한 가설 작성 |
| 필수 2 | 자르기·반전에서 객체 라벨 점검 → 같은 모델의 입력 크기 변경 → 회복·악화 사례 검토 |
| 독립 과제 | 다른 사진의 자르기 검수 → 공정한 전후 평가와 설정 선택 → 처음 보는 사진으로 최종 검증 |

**실행 준비**　기존 `coco_sample.zip` 또는 `coco_dataset.zip`을 노트북과 같은 폴더에 둡니다. Python 3.10 이상, Jupyter, 호환되는 PyTorch·torchvision, pandas, NumPy, Pillow가 필요합니다. 제작 시 CPU에서 torch 2.6.0, torchvision 0.21.0으로 실행했습니다. 첫 사용 시 COCO 사전학습 Mask R-CNN 가중치 약 170 MB를 내려받고 이후 캐시에서 읽습니다. 정답 노트북에는 측정한 출력과 그림이 저장돼 있습니다.

**실험 설계**　제공된 COCO val2017 표본에서 사진 ID 기준으로 Train 12장, Validation 14장, Test 12장을 나눕니다. 평가 대상은 얼룩말·고양이·개이고, 학습 시에는 선택한 사진의 모든 개별 객체 주석을 사용합니다. Train에서 출력층을 한 번 미세조정하고, 그 가중치를 두 평가 설정이 공유합니다. 바꾸는 것은 추론 시 짧은 변의 목표 길이(320 또는 640)입니다. 긴 변 상한은 모두 960입니다. 점수 0.50 이상, 같은 클래스, 박스 IoU 0.50 이상을 정답 검출의 조건으로 사용합니다. 예측 마스크는 픽셀 확률 0.50에서 이진화합니다.

**설정 선택 규칙**　목표는 미검출 감소입니다. Validation에서 Recall이 엄격히 오르고 Precision 하락이 **5%p 이내**이면 640을 선택합니다. 나머지 경우에는 320을 유지합니다. Test는 이 결정을 내린 다음에 엽니다. 선택 규칙과 관심 클래스는 평가 전에 정한 수업용 조건입니다.

공통 준비 셀을 모두 실행한 뒤 각 절을 진행하세요. 독립 과제는 필수 1·2의 답안 변수 없이도 실행할 수 있습니다. 준비 셀은 모델 학습까지 제공합니다. 수강생은 문항에서 판정·집계·결정 근거를 직접 작성합니다.

### 공통 준비 · 사진·주석 읽기

In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

data_path = Path("coco_dataset.zip")
if not data_path.is_file():
    data_path = Path("coco_dataset.zip")
if not data_path.is_file():
    raise FileNotFoundError("노트북 옆에 coco_sample.zip 또는 coco_dataset.zip을 두세요.")
archive = ZipFile(data_path)
objects = pd.read_csv(archive.open("objects.csv"))
files = set(archive.namelist())
usable = objects.loc[objects.image_path.isin(files) & objects.mask_path.isin(files)]

def read_instance(annotation_id):
    row = usable.loc[usable.annotation_id == annotation_id].iloc[0]
    with Image.open(BytesIO(archive.read(row.image_path))) as src:
        photo = np.asarray(src.convert("RGB")).copy()
    with Image.open(BytesIO(archive.read(row.mask_path))) as src:
        mask = np.asarray(src.convert("L")) == 255
    assert photo.shape[:2] == mask.shape
    return photo, mask

def show_pair(photo, mask):
    # 왼쪽은 사진, 오른쪽은 대상 객체를 초록색으로 표시한 사진입니다.
    overlay = photo.copy()
    overlay[mask] = (photo[mask] * 0.45 + np.array([35,225,95]) * 0.55).astype("uint8")
    pair = Image.fromarray(np.concatenate([photo, overlay], axis=1))
    pair.thumbnail((840,390))
    display(pair)
zebras = usable.loc[(usable.image_id == 11760) & (usable.iscrowd == 0)].sort_values("annotation_id")
zebra_photo, zebra_mask = read_instance(589666)
cat_photo, cat_mask = read_instance(46699)

### 공통 준비 · 사진 단위 분할과 텐서 변환

In [2]:
import torch
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.transforms.functional import pil_to_tensor
from torchvision.ops import box_iou

torch.set_num_threads(2)
SEED = 42
TARGETS = ["zebra", "cat", "dog"]
BEFORE, AFTER = 320, 640
SCORE, MATCH_IOU, MASK_CUTOFF = 0.50, 0.50, 0.50
rng = np.random.default_rng(SEED)
train_ids, val_ids, test_ids = [], [11760, 22892], []
used = set(val_ids)
# 정답을 누락하지 않도록 이미지 단위로 모든 주석이 사용 가능한지 확인합니다.
complete = (objects.image_path.isin(files) & objects.mask_path.isin(files)
            & objects.iscrowd.eq(0) & (objects.xmax > objects.xmin) & (objects.ymax > objects.ymin))
eligible_ids = set(objects.image_id) - set(objects.loc[~complete,"image_id"])
for name in TARGETS:
    pool = sorted((set(usable.loc[usable.class_name == name,"image_id"]) & eligible_ids) - used)
    picked = rng.choice(pool, size=12, replace=False).tolist()
    train_ids.extend(picked[:4])
    val_ids.extend(picked[4:8])
    test_ids.extend(picked[8:])
    used.update(picked)
assert set(val_ids).issubset(eligible_ids)
assert len(used) == len(train_ids) + len(val_ids) + len(test_ids)
print("Train / Validation / Test 사진", len(train_ids),len(val_ids),len(test_ids))

def sample(image_id):
    rows = usable.loc[usable.image_id == image_id].sort_values("annotation_id")
    with Image.open(BytesIO(archive.read(rows.iloc[0].image_path))) as im:
        image = im.convert("RGB")
    return image, rows

def mask_of(row):
    with Image.open(BytesIO(archive.read(row.mask_path))) as im:
        return np.asarray(im.convert("L")) == 255

def to_input(image):
    return pil_to_tensor(image).float()/255

def target_for(rows):
    boxes = torch.tensor(rows[["xmin","ymin","xmax","ymax"]].to_numpy(),dtype=torch.float32)
    labels = torch.tensor(rows.category_id.tolist(),dtype=torch.long)
    masks = torch.from_numpy(np.stack([mask_of(row) for row in rows.itertuples()])).to(torch.uint8)
    return {"boxes": boxes, "labels": labels, "masks": masks}

Train / Validation / Test 사진 12 14 12


### 공통 준비 · 마지막 박스·마스크 출력층 학습 함수

In [3]:
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
names = weights.meta["categories"]
def fit_model():
    torch.manual_seed(SEED)
    model = maskrcnn_resnet50_fpn(weights=weights,min_size=BEFORE,max_size=960,
                                  box_score_thresh=0.10,box_detections_per_img=30)
    model.requires_grad_(False)
    box_head = model.roi_heads.box_predictor
    mask_head = model.roi_heads.mask_predictor.mask_fcn_logits
    box_head.requires_grad_(True)
    mask_head.requires_grad_(True)
    optim = torch.optim.SGD([{"params":box_head.parameters(),"lr":0.0002},
                             {"params":mask_head.parameters(),"lr":0.002}])
    old_box = box_head.cls_score.weight.detach().clone()
    old_mask = mask_head.weight.detach().clone()
    history = []
    model.train()
    for image_id in train_ids:
        photo, rows = sample(image_id)
        optim.zero_grad()
        losses = model([to_input(photo)], [target_for(rows)])
        loss = losses["loss_classifier"] + losses["loss_box_reg"] + losses["loss_mask"]
        loss.backward()
        optim.step()
        history.append([image_id,len(rows),float(loss.detach())])
    changes = [(box_head.cls_score.weight.detach()-old_box).abs().max().item(),
               (mask_head.weight.detach()-old_mask).abs().max().item()]
    return model.eval(), pd.DataFrame(history,columns=["image_id","objects","loss"]), changes

### 공통 준비 · 동일 클래스 일대일 대응과 평가 함수

In [4]:
@torch.inference_mode()
def predict(model, image, short_side):
    model.transform.min_size = (short_side,)
    out = model([to_input(image)])[0]
    candidates = []
    for rank,(label,score) in enumerate(zip(out["labels"].tolist(),out["scores"].tolist())):
        if names[label] in TARGETS:
            candidates.append({"name":names[label], "rank":rank+1, "score":score,
                               "box":out["boxes"][rank],
                               "mask":out["masks"][rank,0].numpy()>=MASK_CUTOFF})
    return candidates

def audit_one(img_id, gt, predictions):
    gt = gt[gt.class_name.isin(TARGETS)].reset_index(drop=True)
    gt_boxes = torch.tensor(gt[["xmin","ymin","xmax","ymax"]].to_numpy(),dtype=torch.float32)
    used = set()
    records = []
    for p in predictions:  # 모델의 점수 내림차순
        overlap = box_iou(p["box"][None],gt_boxes)[0].numpy()
        same = gt.class_name.to_numpy() == p["name"]
        free = np.array([i not in used for i in range(len(gt))],dtype=bool)
        good = overlap >= MATCH_IOU
        matches = np.flatnonzero(same & free & good)
        hit, mask_iou = None,np.nan
        if len(matches):
            pos = int(matches[overlap[matches].argmax()])
            used.add(pos)
            hit = int(gt.iloc[pos].annotation_id)
            true_mask = mask_of(gt.iloc[pos])
            mask_iou = (true_mask & p["mask"]).sum()/(true_mask | p["mask"]).sum()
            reason = "TP"
        else:
            pos = int(overlap.argmax())
            if np.any(same & good):
                reason = "중복"
            elif np.any(~same & good):
                reason = "클래스 오류"
            elif np.any(same & (overlap > 0)):
                reason = "위치 오류"
            else:
                reason = "기타 오검출"
        records.append([img_id,p["rank"],p["name"],p["score"],hit,reason,float(overlap[pos]),mask_iou])
    return records

def evaluate(model, ids, size):
    records,truth = [],[]
    for img_id in ids:
        im, rows = sample(img_id)
        gt = rows[rows.class_name.isin(TARGETS)]
        assert len(gt)
        records.extend(audit_one(img_id,gt,predict(model,im,size)))
        for row in gt.itertuples():
            fraction = mask_of(row).sum()/(im.width*im.height)
            truth.append([img_id,row.annotation_id,row.class_name,fraction])
    log = pd.DataFrame(records,columns=["image_id","rank","name","score","hit_id","reason","box_iou","mask_iou"])
    truth = pd.DataFrame(truth,columns=["image_id","annotation_id","name","area_fraction"])
    kept = log[log.score >= SCORE]
    truth["found"] = truth.annotation_id.isin(kept.hit_id.dropna())
    return log,truth

def summary(result):
    log,truth = result
    kept = log.loc[log.score >= SCORE]
    tp = int(kept.hit_id.notna().sum())
    fp,fn = len(kept)-tp,len(truth)-tp
    aps = []
    for name in TARGETS:
        ranked = log.loc[log.name==name].sort_values("score",ascending=False,kind="stable")
        cumulative = ranked.hit_id.notna().cumsum().to_numpy(dtype=float)
        precision = cumulative/np.arange(1,len(ranked)+1)
        recall = cumulative/(truth.name==name).sum()
        aps.append(np.mean([precision[recall>=t-1e-12].max(initial=0) for t in np.linspace(0,1,101)]))
    return pd.Series({"TP":tp,"FP":fp,"FN":fn,"precision":tp/len(kept) if len(kept) else np.nan,
                      "recall":tp/len(truth),"mAP50":np.mean(aps),"mask_iou_matched":kept.mask_iou.mean()})

def choose(validation):
    before,after = validation[BEFORE],validation[AFTER]
    eligible = (after.precision >= before.precision-0.05) and (after.recall > before.recall)
    return AFTER if eligible else BEFORE

### 공통 준비 · 박스와 마스크 시각화 함수

In [5]:
def mask_box(mask):
    rr,cc = np.nonzero(mask)
    if not cc.size:
        return None
    return (int(cc.min()),int(rr.min()),int(cc.max()+1),int(rr.max()+1))

def panel(images, captions):
    thumbs = []
    for image in images:
        thumb = image.copy()
        thumb.thumbnail((300,245))
        thumbs.append(thumb)
    sheet = Image.new("RGB",(300*len(thumbs),282),"white")
    pen = ImageDraw.Draw(sheet)
    for i,(thumb,title) in enumerate(zip(thumbs,captions)):
        sheet.paste(thumb,(300*i,30))
        pen.text((300*i+5,8),title,fill="black")
    display(sheet)

def draw_scene(img_id, short_side):
    photo,rows = sample(img_id)
    gt = rows[rows.class_name.isin(TARGETS)]
    marked = photo.copy()
    pen = ImageDraw.Draw(marked)
    for r in gt.itertuples():
        pen.rectangle((r.xmin,r.ymin,r.xmax,r.ymax),outline="lime",width=3)
        pen.text((r.xmin,r.ymin),f"{r.class_name} #{r.annotation_id}",fill="white",stroke_width=1,stroke_fill="black")
    pred_view = photo.copy()
    pen = ImageDraw.Draw(pred_view)
    for p in predict(model,photo,short_side):
        if p["score"] >= SCORE:
            x1,y1,x2,y2 = p["box"].tolist()
            pen.rectangle((x1,y1,x2,y2),outline="darkorange",width=3)
            pen.text((x1,y1),f'{p["name"]} {p["score"]:.2f}',fill="white",stroke_width=1,stroke_fill="black")
    return marked,pred_view

In [6]:
model, history, delta = fit_model()
print("학습 완료 · 사진",len(history),"장")
val_cache = {}
def validation_at(size):
    if size not in val_cache:
        val_cache[size] = evaluate(model,val_ids,size)
    return val_cache[size]

def paired_objects():
    key = ["image_id","annotation_id","name"]
    tables = []
    for size in (BEFORE,AFTER):
        _,targets = validation_at(size)
        tables.append(targets[key+["found"]].rename(columns={"found":size}))
    return tables[0].merge(tables[1],on=key,validate="one_to_one")

학습 완료 · 사진 12 장


## 필수 1 · 어떤 사진에서 무엇을 틀렸는지 찾기

### 1-1. 실제 학습과 평가 데이터의 경계 확인

세 세트의 사진 수·교집합을 확인하세요. 학습 표에서 첫 사진의 객체 수, 총 학습 객체 수, 가중치 변화를 구하고 사진 한 장이 학습에 들어가는 과정을 설명하세요.

In [ ]:
# 1-1 실습 코드
# 도움말 · history 행 하나는 사진 한 장이자 한 번의 가중치 갱신입니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 1-2. 같은 얼룩말 세 마리에 실제 예측 연결하기

11760번 사진에서 정답 박스·주석 ID와 320 입력의 예측을 나란히 표시하세요. 왼쪽·중앙·오른쪽 개체가 어느 예측 순위와 연결됐는지 표로 확인하고, 중복 후보가 생기면 어떻게 셀지 설명하세요.

In [ ]:
# 1-2 실습 코드
# 도움말 · 정답 행은 xmin으로, 예측은 점수로 정렬됩니다. 화면의 왼쪽과 예측 1위가 같은지 확인하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 1-3. 예측 표의 FP와 정답 표의 FN 분리해서 세기

320 Validation에서 점수 0.50 이상 후보를 판정 이유별로 세고, 정답 표에서 미검출을 뽑으세요. Precision·Recall을 손으로 검산하고 가장 먼저 살펴볼 사진 하나를 고르세요.

In [ ]:
# 1-3 실습 코드
# 도움말 · 분류가 잘못된 예측은 FP입니다. 올바른 후보가 또 없다면 같은 장면에 찾지 못한 정답도 남습니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 1-4. 작은 객체에서 미검출이 많은지 분모까지 비교하기

정답 마스크 면적이 사진의 5% 미만인 집단과 그 밖의 집단에서 정답 수·미검출 수·미검출률을 비교하세요. 관측에서 나온 가설과 추가로 확인할 장면 조건을 구분하세요.

In [ ]:
# 1-4 실습 코드
# 도움말 · 같은 “2건”이라도 정답 4개에서 나온 것과 15개에서 나온 것은 다릅니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

## 필수 2 · 전처리의 유효성을 검사하고 오류 원인에 맞는 변경 실험하기

### 2-1. 같은 반쪽 사진이어도 원래 객체 라벨을 유지할 수 있을까?

주석 589666의 사진·마스크에 같은 왼쪽 또는 오른쪽 절반 자르기를 적용하세요. 남은 픽셀 수·보존율·새 마스크 박스를 구하고, 어느 창에서 해당 주석을 학습 예제로 유지할지 쓰세요.

In [ ]:
# 2-1 실습 코드
# 도움말 · mask_box는 전경 픽셀을 감싸는 정수 XYXY를 반환합니다. 오른쪽·아래 끝은 포함하지 않으며, 전경이 없으면 None입니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 2-2. 픽셀 수가 보존돼도 라벨이 틀릴 수 있는가?

얼룩말 사진을 좌우 반전하세요. 마스크도 반전한 경우와 원래 마스크를 잘못 붙인 경우를 나란히 표시하세요. 면적·마스크 박스·올바른 마스크와 잘못 붙인 마스크의 IoU를 구하고 검수 기준을 제안하세요.

In [ ]:
# 2-2 실습 코드
# 도움말 · 색상 배열은 (행, 열, 채널), 마스크는 (행, 열)입니다. 두 배열 모두 열 축의 순서를 바꿉니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 2-3. 입력을 키운 뒤 새로 찾은 객체와 놓치게 된 객체 추적하기

같은 모델로 Validation의 입력 짧은 변을 640으로 바꾸세요. 320·640의 정답 표를 객체 ID로 결합해 회복·악화·계속 검출·계속 미검출을 각각 세세요. 회복 사례와 악화 사례를 한 장씩 골라 실제 예측 그림을 비교하세요.

In [ ]:
# 2-3 실습 코드
# 도움말 · paired_objects는 (image_id, annotation_id, name)으로 두 정답 표를 결합합니다. 두 found 열을 비교하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 2-4. 작은 물체와 어두운 장면에는 어떤 후속 실험이 맞을까?

두 입력 크기의 작은 객체 집계로 해상도 가설을 평가하세요. 어두운 사진에서 오류가 반복될 때의 증강 실험, Feature Extraction에서 Fine-tuning으로 전환하는 실험을 각각 제안하세요. 무엇을 고정하고 무엇을 검증할지도 쓰세요.

In [ ]:
# 2-4 실습 코드
# 도움말 · 입력 크기·밝기·학습 범위를 동시에 바꾸면 결과를 어느 변경에 돌릴지 판단하기 어렵습니다.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

## 과제 1 · 새로운 이미지의 라벨을 검수하고 개선안을 결정하기

### 3-1. 같은 300열에 고양이가 얼마나 남는가?

사진 22892, 주석 46699의 원본·왼쪽 300열·오른쪽 300열을 비교하세요. 원본 대비 개체 픽셀 보존율과 새 마스크 박스를 계산하고, 고양이 전체 형태를 보존하는 학습 예제로 어느 창을 선택할지 근거를 쓰세요.

In [ ]:
# 3-1 실습 코드
# 도움말 · 두 창 모두 같은 폭입니다. 각 창의 전경 수를 원본 고양이 전경 수로 나누세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 3-2. 검출 수·오류 종류·지표의 변화를 한 표로 비교하기

동일한 Validation에서 After − Before를 계산하세요. TP·FP·FN은 건, Precision·Recall·mAP50·매칭 마스크 IoU는 0~1 값을 100배 한 척도의 점수 차로 표시하세요. FP 총수와 FP 유형이 함께 어떻게 바뀌었는지 근거를 쓰세요.

In [ ]:
# 3-2 실습 코드
# 도움말 · summary의 mAP50은 후보 순위를 사용합니다. 고정 문턱 Precision 한 점과 구분하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 3-3. Validation에서 결정한 설정을 잠근 뒤 Test 확인하기

사전에 제시한 규칙으로 입력 크기를 선택한 후 Test에 기준 설정과 선택 설정을 적용하세요. 두 설정이 같으면 한 번만 실행하세요. Test의 미검출 감소와 추가 오검출을 함께 보고하고 Validation에서 본 변화와 비교하세요.

In [ ]:
# 3-3 실습 코드
# 도움말 · 변경 설정은 Recall이 더 높고 Precision 하락이 5%p 이내여야 합니다. test_ids는 결정 이후에 사용하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

### 3-4. 제조 사례와 실제 COCO 결과로 서로 다른 결론 쓰기

교안 제조 사례의 지표·오류 건수 변화량을 구해 COCO 실측과 구분하세요. 서로 다른 Validation 사진 200장에서 mAP50 0.75와 0.82가 나온 두 모델의 우열을 판단할 수 있는지, 공정한 재평가 조건과 함께 답하세요.

In [ ]:
metrics = pd.DataFrame({"Metric":["Precision","Recall","mAP@0.5"],
                        "Before":[0.85,0.60,0.68], "After":[0.78,0.78,0.74]})
cases = pd.DataFrame({"유형":["작은 균열 미검출","잘못된 균열 검출","위치 오류"],
                      "Before":[20,5,7], "After":[8,9,4]})

# 3-4 실습 코드
# 도움말 · 제조 사례 원자료는 아래 제공됩니다. 단위가 다른 두 표를 분리해서 계산하세요.
# 여기에 풀이를 작성하세요.

**나의 답안**

- 코드 출력 또는 그림에서 확인한 근거
- 이 근거로 내릴 수 있는 판단

## 실습 마무리

1. **어떤 문제가 있었나요?** 실측된 오류 한 건과 증강에서 생길 수 있는 라벨 오류 한 건을 주석 번호 또는 사진 ID로 설명하세요.
2. **무엇을 바꾸고 무엇을 고정했나요?** 자르기 예제와 모델 비교 실험을 나누어 쓰세요.
3. **어디까지 개선됐나요?** Validation에서 설정을 고른 근거와 별도 Test에서 확인한 이득·비용을 분자·분모 또는 실제 건수로 쓰세요.